# PART 7-3. 리뷰 감성 분석 ③ 사전학습 모델 (BERT, Hugging Face)

> **핵심 질문: 이미 학습된 AI를 가져다 쓰면?**

## 📋 한눈에 보기

**작업 흐름**
```
1. 7-1과 같은 리뷰 준비
2. 사전학습 BERT 불러오기 (pipeline)
3. 리뷰 몇 개로 먼저 테스트
4. 같은 test 500개로 7-1 · 7-2 · BERT 비교
5. 사전학습 모델의 장단점 정리
```

| 항목 | 내용 |
|---|---|
| 사용 데이터 | 7-1과 같은 test 중 500건 |
| 결과물 | 저장 없음 (배포하지 않음) |

**🎯 핵심 (꼭 가져갈 것)**
- 사전학습 모델이 항상 더 좋은 것은 아닙니다. **같은 test로 직접 비교**해야 합니다.
- BERT는 학습 없이 바로 쓰고 여러 언어를 지원하지만, 용량이 크고(약 540MB) 우리 데이터로 학습하지 않았습니다.

**💡 인사이트 (이 파트에서 확인한 사실)**
- 학습 없이 쓴 BERT 정확도 0.730 (중립 답변 7% 포함) vs 7-1 머신러닝 0.918
- 이 데이터에서는 **가볍고 정확한 7-1 모델을 배포**합니다. BERT를 이 데이터로 추가 학습(파인튜닝)하면 더 오를 수 있습니다(심화).

---

## 수업 목표
- 이미 대용량 데이터로 사전학습된 BERT 모델을 가져와서 감성분석을 수행합니다.
- 직접 학습한 모델(PART 7-1 머신러닝, PART 7-2 딥러닝)과 사전학습 모델의 차이를 **같은 test**로 비교합니다.

## 수업 진행 포인트
| 구분 | 설명 |
|---|---|
| 데이터 관점 | 같은 리뷰 데이터를 사용합니다. |
| 코드 관점 | Hugging Face pipeline 사용법을 익힙니다. |
| AI 관점 | 사전학습 모델의 장점(학습 없이 바로 사용, 다국어)과 한계(우리 데이터에 맞춰 학습되지 않음)를 이해합니다. |


## 📌 이 파트에서 사용하는 API 흐름 (HuggingFace)

| 단계 | 함수 | 역할 |
|---|---|---|
| ① | pipeline() | 모델 로드 + 전처리 + 예측 한번에 |
| ② | predict_bert() | 예측 실행 |

> BERT는 사전학습 모델 → Define / Compile / Fit 없이 바로 사용


## PART 7-2 vs PART 7-3 비교

| 항목 | PART 7-2 (직접 학습한 딥러닝) | PART 7-3 (BERT) |
|---|---|---|
| 학습 방식 | 직접 학습 | 사전학습 모델 사용 |
| Tokenizer | 직접 만들기 | 모델 내부 포함 |
| 언어 지원 | 학습 데이터 언어만 | 다국어 (한국어 포함) |
| 성능 | 이 데이터로 직접 측정 (test 정확도 약 90~92%) | 같은 test 500개로 정확도 측정 (셀 11) |
| 코드 복잡도 | 높음 | 낮음 |

> ⚠️ "사전학습 모델이 항상 더 좋다"는 보장은 없습니다.
> 이 모델은 일반 문장으로 학습된 다국어 모델이라, Olist 리뷰처럼 특정 분야 문장에서는 직접 학습한 모델보다 낮을 수도 있습니다.
> 성능은 반드시 **같은 test 데이터로 정확도를 계산해 비교**해야 합니다.

In [2]:
# 셀 1. Google Drive 연결

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 셀 2. Hugging Face 라이브러리 설치
# 처음 실행 시에만 설치가 필요합니다.

!pip install transformers torch

#!pip install transformers torch -q
# -q: 설치 로그 줄이기

In [4]:
# 셀 3. 라이브러리 불러오기

import pandas as pd
from transformers import pipeline
# pipeline: 모델 불러오기 + 전처리 + 예측을 한 번에 처리해주는 함수

base_path = "/content/drive/MyDrive/Colab Notebooks/Data_Analysis/Olist"
print("라이브러리 불러오기 완료!")

라이브러리 불러오기 완료!


In [5]:
# 셀 4. 데이터 불러오기 및 전처리

df = pd.read_csv(f"{base_path}/data/olist_master_data.csv")

# 통합 데이터는 주문 1건이 여러 행이라 같은 리뷰가 여러 번 들어 있습니다.
# review_id 기준으로 중복을 지우고 리뷰 1개 = 1행으로 사용합니다.
review_df = df.drop_duplicates("review_id")[["review_comment_message", "review_score"]].copy()
review_df = review_df.dropna()

# 이진 분류(Binary Classification)를 위해 중립(3점) 리뷰는 제거
review_df = review_df[review_df["review_score"] != 3]

review_df["label"] = review_df["review_score"].apply(
    lambda s: 1 if s >= 4 else 0
)

print(f"분석할 리뷰 수: {len(review_df):,}")
review_df.head()

분석할 리뷰 수: 37,139


,review_comment_message,review_score,label
0,"Não testei o produto ainda, mas ele veio corre...",4.0,1
3,Muito bom o produto.,4.0,1
5,O produto foi exatamente o que eu esperava e e...,5.0,1
8,fiquei triste por n ter me atendido.,2.0,0
13,Aguardando retorno da loja,1.0,0



### 🔵 ① pipeline() → 모델 로드


In [6]:
# 셀 5. BERT 감성분석 모델 불러오기
# 다국어 감성분석 모델 (포르투갈어, 한국어 등 지원)
# 처음 실행 시 모델 파일 다운로드 (약 540MB, 1~2분 소요)

# Tokenizer 불러오기 + 단어 → 숫자 변환 + BERT 모델 불러오기 + 감성분석
sentiment_model = pipeline(
    "sentiment-analysis",
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student"
)

print("모델 불러오기 완료!")

config.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  541MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.92M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

모델 불러오기 완료!


```
BERT 계열 모델은 우리가 아까처럼 새로 단어집을 만드는 게 아니라,
이미 학습되어 있는 Tokenizer와 단어집을 같이 불러와서 사용


문장 입력
   ↓
Tokenizer
   ↓
단어/부분단어로 분리
   ↓
미리 만들어진 단어집 기준으로 숫자 ID 변환
   ↓
BERT 모델 입력
   ↓
긍정(positive) / 중립(neutral) / 부정(negative) 3가지로 판단
```

| 직접 만든 모델                 | BERT 사전학습 모델            |
| ------------------------ | ----------------------- |
| Tokenizer() 직접 생성      | Tokenizer도 이미 준비됨       |
| fit_on_texts()로 단어집 생성 | 이미 학습된 단어집 사용           |
| pad_sequences() 직접 처리  | 내부에서 입력 형태 맞춤           |
| 모델 직접 학습                 | 이미 학습된 모델 사용            |
| .keras, .pkl 직접 저장   | Hugging Face 모델을 불러와 사용 |



> 📌 우리 라벨은 긍정/부정 2가지인데, 이 모델은 **중립(neutral)까지 3가지**로 답합니다.
> 그래서 AI 예측이 neutral이면 실제 라벨과 1:1로 비교할 수 없다는 점에 주의합니다.

In [7]:
# 셀 6. 리뷰 1개로 먼저 테스트

test_text = "배송이 빠르고 상품이 좋아요"
result = sentiment_model(test_text)

print(f"입력: {test_text}")
print(f"결과: {result}")
# ▶ label = positive / neutral / negative (3가지), score = 신뢰도(0~1)

입력: 배송이 빠르고 상품이 좋아요
결과: [{'label': 'positive', 'score': 0.8837409615516663}]



### 🔵 ② predict_bert() → 예측 실행


In [8]:
# 셀 7. 감성분석 함수 만들기

def predict_bert(text):
    """
    BERT 모델로 문장의 감성을 예측합니다.
    BERT는 한 번에 최대 512개 토큰(단어 조각)까지만 처리할 수 있습니다.
    truncation=True : 512 토큰을 넘으면 뒷부분을 잘라서 처리합니다.
    """
    result = sentiment_model(text, truncation=True)[0]
    return result["label"], round(result["score"], 3)

# 테스트
print(predict_bert("produto muito bom gostei"))  # 좋은 제품이에요
print(predict_bert("produto ruim nao gostei"))   # 나쁜 제품이에요
print(predict_bert("배송이 빠르고 상품이 좋아요"))

('positive', 0.638)
('negative', 0.5)
('positive', 0.884)


In [9]:
# 셀 8. 샘플 리뷰 20개에 BERT 적용 (약 30초)

sample_df = review_df.sample(20, random_state=42).copy()

sample_df["ai_result"] = sample_df["review_comment_message"].apply(predict_bert)
sample_df["ai_label"]  = sample_df["ai_result"].apply(lambda x: x[0])
sample_df["ai_score"]  = sample_df["ai_result"].apply(lambda x: x[1])

result_df = sample_df[[
    "review_comment_message", "review_score", "label", "ai_label", "ai_score"
]]
result_df

,review_comment_message,review_score,label,ai_label,ai_score
21879,Tudo correu bem. O produto chegou antes do pra...,4.0,1,positive,0.597
17122,"essa loja vendeu o produto, mas nao tinha em e...",1.0,0,negative,0.416
62568,comprei 2 toner hp ce278a e só recebi 1 e aind...,2.0,0,positive,0.388
113055,PENDENTE LINDO! TA DE PARABÈNS AS lannister PO...,5.0,1,positive,0.552
79956,Ainda nao instalei recebi ontem,5.0,1,negative,0.459
42410,Muito bom,5.0,1,negative,0.679
100848,Demora a entrega do produto,1.0,0,positive,0.666
101269,Empresa responsável é comprometida com a entre...,5.0,1,positive,0.580
59734,demorou demais para me avisar que o produto es...,1.0,0,positive,0.532
1554,Produto excelente.,5.0,1,positive,0.982


In [10]:
# 셀 9. 실제 평점 vs AI 예측 비교 출력
# for i in range(10, 15):
for i in range(5):
    row = result_df.iloc[i]
    print(f"\n[리뷰 {i+1}]")
    print(f"  문장:      {str(row['review_comment_message'])[:60]}...")
    print(f"  실제 평점: {row['review_score']}점")
    print(f"  실제 라벨: {'긍정' if row['label']==1 else '부정'}")
    print(f"  AI 예측:   {row['ai_label']}")
    print(f"  AI 신뢰도: {row['ai_score']:.1%}")


[리뷰 1]
  문장:      Tudo correu bem. O produto chegou antes do prazo informado e...
  실제 평점: 4.0점
  실제 라벨: 긍정
  AI 예측:   positive
  AI 신뢰도: 59.7%

[리뷰 2]
  문장:      essa loja vendeu o produto, mas nao tinha em estoque pra ent...
  실제 평점: 1.0점
  실제 라벨: 부정
  AI 예측:   negative
  AI 신뢰도: 41.6%

[리뷰 3]
  문장:      comprei 2 toner hp ce278a e só recebi 1 e ainda emitiram a n...
  실제 평점: 2.0점
  실제 라벨: 부정
  AI 예측:   positive
  AI 신뢰도: 38.8%

[리뷰 4]
  문장:      PENDENTE LINDO! TA DE PARABÈNS AS lannister POR VENDER UM PR...
  실제 평점: 5.0점
  실제 라벨: 긍정
  AI 예측:   positive
  AI 신뢰도: 55.2%

[리뷰 5]
  문장:      Ainda nao instalei recebi ontem...
  실제 평점: 5.0점
  실제 라벨: 긍정
  AI 예측:   negative
  AI 신뢰도: 45.9%


In [11]:
# 셀 10. 랜덤 5개 리뷰 출력
sample_df = result_df.sample(5, random_state=0)   # random_state : 실행할 때마다 같은 5개가 나오도록 고정

for i, row in sample_df.iterrows():

    print(f"\n[리뷰 {i}]")
    print(f"  문장:      {str(row['review_comment_message'])[:60]}...")
    print(f"  실제 평점: {row['review_score']}점")
    print(f"  실제 라벨: {'긍정' if row['label']==1 else '부정'}")
    print(f"  AI 예측:   {row['ai_label']}")
    print(f"  AI 신뢰도: {row['ai_score']:.1%}")


[리뷰 64765]
  문장:      Até o presente momento foi recebido apenas a base do bebê co...
  실제 평점: 1.0점
  실제 라벨: 부정
  AI 예측:   positive
  AI 신뢰도: 57.7%

[리뷰 17122]
  문장:      essa loja vendeu o produto, mas nao tinha em estoque pra ent...
  실제 평점: 1.0점
  실제 라벨: 부정
  AI 예측:   negative
  AI 신뢰도: 41.6%

[리뷰 2848]
  문장:      Meu produto n chegou ainda ao destino, ainda n recebi n poss...
  실제 평점: 1.0점
  실제 라벨: 부정
  AI 예측:   positive
  AI 신뢰도: 35.6%

[리뷰 59734]
  문장:      demorou demais para me avisar que o produto está indisponíve...
  실제 평점: 1.0점
  실제 라벨: 부정
  AI 예측:   positive
  AI 신뢰도: 53.2%

[리뷰 15777]
  문장:      Ainda não avaliei o produto, porém se cumprir as funções pre...
  실제 평점: 5.0점
  실제 라벨: 긍정
  AI 예측:   negative
  AI 신뢰도: 41.9%


---
## 📌 같은 시험지로 비교하기 (PART 7-1 머신러닝 vs PART 7-2 딥러닝 vs PART 7-3 BERT)

위에서는 20개를 눈으로만 확인했습니다. 성능을 비교하려면 **같은 test 데이터로 정확도를 계산**해야 합니다.
PART 7-1·7-2와 똑같은 방법(random_state=42)으로 test를 나누고, 그중 500개로 세 모델을 비교합니다.

> 📌 이 BERT 모델은 **positive / neutral / negative 3가지**로 답합니다. 우리 정답은 긍정/부정 2가지라서
> neutral은 따로 세고, ① neutral을 오답으로 본 정확도 ② neutral을 뺀 정확도를 함께 봅니다.

In [12]:
# 셀 11. 같은 test 데이터 500개로 PART 7-1 · PART 7-2 · BERT 비교 (약 1~2분)
import pickle
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ① PART 7-1·7-2와 똑같이 나누기 → 같은 test 데이터
_, X_test, _, y_test = train_test_split(
    review_df["review_comment_message"], review_df["label"],
    test_size=0.2, random_state=42, stratify=review_df["label"]
)
eval_df = pd.DataFrame({"text": X_test, "label": y_test}).sample(500, random_state=42)

# ② BERT 예측 (여러 문장을 한 번에 넣으면 더 빠릅니다)
bert_out = sentiment_model(eval_df["text"].tolist(), truncation=True, batch_size=32)
eval_df["bert"] = [r["label"] for r in bert_out]                       # positive / neutral / negative
label_map = {"positive": 1, "negative": 0}

# ③ PART 7-1에서 저장한 머신러닝 모델로 예측
ml = joblib.load(f"{base_path}/model/part7_1_sentiment_ml.pkl")
eval_df["ml"] = ml["model"].predict(ml["tfidf"].transform(eval_df["text"]))

# ④ PART 7-2에서 저장한 딥러닝 모델로 예측
#    PART 7-2와 똑같이 MAX_LEN = 50으로 패딩해야 합니다. (길이가 다르면 예측이 틀어집니다)
keras_model = load_model(f"{base_path}/model/part7_2_sentiment_model.keras")
with open(f"{base_path}/model/part7_2_tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)
seq = pad_sequences(tokenizer.texts_to_sequences(eval_df["text"]), maxlen=50, padding="post", truncating="post")
eval_df["keras"] = (keras_model.predict(seq, verbose=0)[:, 0] >= 0.5).astype(int)

# ⑤ 정확도 비교
is_neutral = eval_df["bert"] == "neutral"
bert_pred  = eval_df["bert"].map(label_map)                             # neutral → NaN

print(f"BERT가 neutral로 답한 비율        : {is_neutral.mean():.1%}")
print(f"BERT 정확도 (neutral = 오답)      : {(bert_pred == eval_df['label']).mean():.3f}")
print(f"BERT 정확도 (neutral 제외)        : {(bert_pred[~is_neutral] == eval_df.loc[~is_neutral, 'label']).mean():.3f}")
print(f"PART 7-1 머신러닝 정확도 (같은 500개): {accuracy_score(eval_df['label'], eval_df['ml']):.3f}")
print(f"PART 7-2 딥러닝 정확도 (같은 500개)  : {accuracy_score(eval_df['label'], eval_df['keras']):.3f}")

BERT가 neutral로 답한 비율        : 7.2%
BERT 정확도 (neutral = 오답)      : 0.730
BERT 정확도 (neutral 제외)        : 0.787
PART 7-1 머신러닝 정확도 (같은 500개): 0.918
PART 7-2 딥러닝 정확도 (같은 500개)  : 0.922


**결과 해석 방법**
- PART 7-1·7-2 모델은 **이 데이터(Olist 리뷰)로 직접 학습**했고, BERT 모델은 **일반 문장으로 학습된 다국어 모델을 그대로** 사용했습니다.
- 그래서 이 데이터에서는 직접 학습한 모델이 더 높게 나올 수 있습니다. 사전학습 모델의 장점은 "학습 없이 바로, 여러 언어로" 쓸 수 있다는 것입니다.
- BERT를 이 데이터로 추가 학습(파인튜닝)하면 성능을 더 끌어올릴 수 있습니다. (심화 과정)

> ⚠️ 이 셀의 결과는 실행 환경(모델 버전)에 따라 조금 달라질 수 있으니, 숫자는 직접 실행한 결과로 해석합니다.

---
## 📦 저장 대신 불러오기: 모델 허브(Hugging Face)라는 방식

PART 7-1~7-2에서는 직접 학습한 모델을 **파일로 저장**했습니다. BERT는 저장하지 않습니다. 왜일까요?

| 구분 | PART 7-1~7-2 (직접 학습) | PART 7-3 (사전학습 모델) |
|---|---|---|
| 모델이 있는 곳 | 내 Google Drive (`.pkl`, `.keras`) | Hugging Face **모델 허브** (인터넷) |
| 불러오는 방법 | `joblib.load()`, `load_model()` | `pipeline(..., model="모델 이름")` |
| 전처리 도구 | TF-IDF, Tokenizer를 **직접 저장**해야 함 | 모델 안에 Tokenizer가 **함께 들어 있음** |
| 크기 | 수 KB ~ 수 MB | **약 540MB** |
| 처음 실행할 때 | 바로 불러옴 | 인터넷에서 내려받음 (1~2분) |

> 📌 모델 이름(`lxyuan/distilbert-base-multilingual-cased-sentiments-student`)만 알면 어디서든 같은 모델을 불러올 수 있습니다.
> 대신 **용량이 크고(모델 다운로드 약 540MB) 처음 실행이 느리다**는 점을 STEP2(서빙)·STEP3(배포)에서 고려해야 합니다.
> 이 데이터에서는 PART 7-1 머신러닝 모델(약 170KB)이 더 가볍고 정확도도 높기 때문에, **서비스에 배포할 감성분석 모델로는 PART 7-1 모델을 사용**합니다.

## 마무리 정리

| 확인할 내용 | 설명 |
|---|---|
| 이 파트의 핵심 | 사전학습 BERT 모델로 리뷰 감성을 분석합니다. |
| 사전학습 모델의 장점 | 직접 학습 없이 바로 사용, 다국어 지원 |
| 주의 | 성능은 같은 test 데이터로 정확도를 계산해야 비교할 수 있습니다. |
| 모델 허브 | 사전학습 모델은 파일로 저장하지 않고 모델 이름으로 불러옵니다. 크기가 커서 서빙·배포 때 고려가 필요합니다. |
| 다음 단계 | PART 8에서 판매자를 비슷한 유형끼리 묶는 비지도 학습(K-Means)을 진행합니다. |